# Building Autograd from Scratch

A scalar-valued automatic differentiation engine — built step by step, inspired by Karpathy's micrograd.

**What you'll build:**
1. A `Value` class that constructs a computation graph on the fly
2. Reverse-mode autodiff via `_backward` closures + topological sort
3. `tanh`, `relu`, `exp`, `log`, `**` — all with correct gradients
4. `Neuron → Layer → MLP` on top of the engine
5. A training loop that learns XOR

**The one-sentence intuition:** wrap every scalar in a `Value` that remembers its parents;
once you have the graph, walk it in reverse applying the chain rule.

In [4]:
  import sys                                                                              
  !{sys.executable} -m ensurepip --upgrade                                                
  !{sys.executable} -m pip install graphviz    

Looking in links: /var/folders/qz/ycfy4hxs01321rg_7tdl5kqr0000gn/T/tmpjg0o71vn
Processing /var/folders/qz/ycfy4hxs01321rg_7tdl5kqr0000gn/T/tmpjg0o71vn/pip-25.0.1-py3-none-any.whl

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: /Users/hitesh/pytorch_practise/.venv/bin/python3 -m pip install --upgrade pip


In [5]:
# pip install graphviz
# macOS: brew install graphviz  |  Ubuntu: apt install graphviz
import math
import random
import matplotlib.pyplot as plt
from graphviz import Digraph

## Part 1 — The `Value` class (forward pass only)

Every number in the engine is wrapped in a `Value`. It stores:

| field | purpose |
|-------|---------|
| `data` | the scalar number |
| `grad` | ∂L/∂self — filled in by `backward()` |
| `_prev` | the set of `Value`s that produced this node |
| `_op` | string label of the operation (for viz) |
| `_backward` | closure that propagates grad one step (added in Part 3) |

For now `_backward` is a no-op; we're just building the forward graph.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data      = float(data)
        self.grad      = 0.0            # ∂L/∂self — set during backward
        self._backward = lambda: None   # overwritten per op in Part 3
        self._prev     = set(_children)
        self._op       = _op
        self.label     = label

    def __repr__(self):
        return f'Value(data={self.data:.4f}, grad={self.grad:.4f})'

    # ── forward ops ───────────────────────────────────────────────────────────
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), '+')

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), '*')

    def __pow__(self, exp):
        assert isinstance(exp, (int, float))
        return Value(self.data ** exp, (self,), f'**{exp}')

    def tanh(self):
        return Value(math.tanh(self.data), (self,), 'tanh')

    def relu(self):
        return Value(max(0.0, self.data), (self,), 'relu')

    def exp(self):
        return Value(math.exp(self.data), (self,), 'exp')

    def log(self):
        assert self.data > 0, f'log of non-positive: {self.data}'
        return Value(math.log(self.data), (self,), 'log')

    # ── convenience so literals work: 2 + v, 2 * v, -v, v - w, v / w ─────────
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __neg__(self):         return self * -1
    def __sub__(self, other):  return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __truediv__(self, other):  return self * other**-1
    def __rtruediv__(self, other): return other * self**-1

In [ ]:
a = Value(2.0,  label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
f = Value(-2.0, label='f')

e = a * b;  e.label = 'e'   # -6
d = e + c;  d.label = 'd'   # 4
L = d * f;  L.label = 'L'   # -8

print(L)           # grad is 0 — backward hasn't run yet
print('op:', L._op)
print('parents:', L._prev)

## Part 2 — Visualising the computation graph

`trace` walks the DAG collecting all nodes and directed edges.
`draw_dot` renders them with graphviz:

- **Rectangle nodes** — `Value` nodes showing `label | data | grad`
- **Circle nodes** — operation bubbles linking inputs to their output

Returning the `Digraph` object from the last line of a cell renders it inline in Jupyter.

In [ ]:
def trace(root):
    nodes, edges = set(), set()
    def walk(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                walk(child)
    walk(root)
    return nodes, edges


def draw_dot(root, rankdir='LR'):
    dot = Digraph(graph_attr={'rankdir': rankdir, 'size': '12,8'})
    nodes, edges = trace(root)

    for n in nodes:
        uid   = str(id(n))
        parts = ([n.label] if n.label else []) + [f'data {n.data:.4f}', f'grad {n.grad:.4f}']
        dot.node(uid, label='{ ' + ' | '.join(parts) + ' }', shape='record')
        if n._op:
            op_uid = uid + n._op
            dot.node(op_uid, label=n._op, shape='circle')
            dot.edge(op_uid, uid)

    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [ ]:
# L = (a*b + c) * f   — same values as above
a = Value(2.0,  label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
f = Value(-2.0, label='f')
e = a * b;  e.label = 'e'
d = e + c;  d.label = 'd'
L = d * f;  L.label = 'L'

draw_dot(L)  # all grads are 0 — backward hasn't run yet

## Part 3 — Backpropagation

**Chain rule:** if `L = f(g(x))` then `dL/dx = (dL/dg) · (dg/dx)`.

In the graph, every node `out` accumulates `out.grad = dL/d(out)` from its consumers.
Each op only needs to know its *local* derivative; multiplying by `out.grad` gives the input's gradient.

| Operation | Local derivative | `_backward` |
|-----------|-----------------|-------------|
| `z = a + b` | ∂z/∂a = 1, ∂z/∂b = 1 | `a.grad += out.grad` |
| `z = a * b` | ∂z/∂a = b, ∂z/∂b = a | `a.grad += b.data * out.grad` |
| `z = aⁿ` | ∂z/∂a = n·aⁿ⁻¹ | `a.grad += n*a.data**(n-1)*out.grad` |
| `z = tanh(a)` | 1 − tanh²(a) | `a.grad += (1 − t²)*out.grad` |
| `z = relu(a)` | 1 if a>0 else 0 | `a.grad += (out.data>0)*out.grad` |
| `z = exp(a)` | exp(a) | `a.grad += e*out.grad` |
| `z = log(a)` | 1/a | `a.grad += (1/a.data)*out.grad` |

We use `+=` not `=` because the **same** node can appear as input to multiple operations
(e.g. `a * a`), and all paths must add their contribution.

`backward()` topologically sorts the graph (post-order DFS) so each node is always
visited *after* every node that depends on it, then iterates in reverse.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data      = float(data)
        self.grad      = 0.0
        self._backward = lambda: None
        self._prev     = set(_children)
        self._op       = _op
        self.label     = label

    def __repr__(self):
        return f'Value(data={self.data:.4f}, grad={self.grad:.4f})'

    # ── ops with _backward closures ───────────────────────────────────────────
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out   = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad  += out.grad          # ∂(a+b)/∂a = 1
            other.grad += out.grad          # ∂(a+b)/∂b = 1
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out   = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad  += other.data * out.grad   # ∂(a*b)/∂a = b
            other.grad += self.data  * out.grad   # ∂(a*b)/∂b = a
        out._backward = _backward
        return out

    def __pow__(self, exp):
        assert isinstance(exp, (int, float))
        out = Value(self.data**exp, (self,), f'**{exp}')
        def _backward():
            self.grad += exp * (self.data**(exp - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t   = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1.0 - t**2) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), 'relu')
        def _backward():
            self.grad += float(out.data > 0) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        e   = math.exp(self.data)
        out = Value(e, (self,), 'exp')
        def _backward():
            self.grad += e * out.grad       # ∂(eˣ)/∂x = eˣ
        out._backward = _backward
        return out

    def log(self):
        assert self.data > 0, f'log({self.data}) undefined'
        out = Value(math.log(self.data), (self,), 'log')
        def _backward():
            self.grad += (1.0 / self.data) * out.grad
        out._backward = _backward
        return out

    # ── backward: topo sort + chain rule sweep ────────────────────────────────
    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0                     # dL/dL = 1
        for node in reversed(topo):
            node._backward()

    # ── convenience ───────────────────────────────────────────────────────────
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __neg__(self):         return self * -1
    def __sub__(self, other):  return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __truediv__(self, other):  return self * other**-1
    def __rtruediv__(self, other): return other * self**-1

In [ ]:
# Karpathy's classic single-neuron example
# x1,x2 are inputs; w1,w2 are weights; b chosen so tanh(n) ≈ 0.7071
x1 = Value(2.0,   label='x1')
x2 = Value(0.0,   label='x2')
w1 = Value(-3.0,  label='w1')
w2 = Value(1.0,   label='w2')
b  = Value(6.8813735870195432, label='b')

x1w1   = x1 * w1;     x1w1.label   = 'x1*w1'
x2w2   = x2 * w2;     x2w2.label   = 'x2*w2'
preact = x1w1 + x2w2; preact.label = 'x1w1+x2w2'
n      = preact + b;  n.label      = 'n'
o      = n.tanh();    o.label      = 'o'

o.backward()

print(f'o.data = {o.data:.6f}  (expected ≈ 0.707107)')
print()
print('Gradients ∂o/∂x:')
for v in [o, n, b, preact, x1w1, x2w2, x1, w1, x2, w2]:
    print(f'  {v.label:14s}  grad = {v.grad:+.6f}')

In [ ]:
draw_dot(o)   # grad column now shows the backpropagated values

In [ ]:
try:
    import torch
    x1t = torch.tensor(2.0,  dtype=torch.float64, requires_grad=True)
    x2t = torch.tensor(0.0,  dtype=torch.float64, requires_grad=True)
    w1t = torch.tensor(-3.0, dtype=torch.float64, requires_grad=True)
    w2t = torch.tensor(1.0,  dtype=torch.float64, requires_grad=True)
    bt  = torch.tensor(6.8813735870195432, dtype=torch.float64, requires_grad=True)
    ot  = torch.tanh(x1t*w1t + x2t*w2t + bt)
    ot.backward()

    print('PyTorch vs our engine:')
    pairs = [(x1t,x1,'x1'),(x2t,x2,'x2'),(w1t,w1,'w1'),(w2t,w2,'w2'),(bt,b,'b')]
    for pt, ours, name in pairs:
        ok = '✓' if abs(pt.grad.item() - ours.grad) < 1e-9 else '✗'
        print(f'  {name}: pytorch={pt.grad.item():+.6f}  ours={ours.grad:+.6f}  {ok}')
except ImportError:
    print('PyTorch not installed — skipping cross-check')

## Part 4 — Neural network primitives

With autograd working, the network is just arithmetic:

- **`Neuron`** — weighted sum of inputs + bias, then activation
- **`Layer`**  — list of neurons that all read the same input vector
- **`MLP`**    — sequence of layers

`Module.zero_grad()` resets every `.grad` to 0 before each backward pass.
This is necessary because `_backward` uses `+=`, so grads accumulate across calls.

In [ ]:
class Module:
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0
    def parameters(self):
        return []


class Neuron(Module):
    def __init__(self, n_in, activation='tanh'):
        self.w          = [Value(random.uniform(-1, 1)) for _ in range(n_in)]
        self.b          = Value(0.0)
        self.activation = activation

    def __call__(self, x):
        # dot product then activation
        act = sum((wi * xi for wi, xi in zip(self.w, x)), Value(0.0)) + self.b
        if self.activation == 'tanh':  return act.tanh()
        if self.activation == 'relu':  return act.relu()
        return act   # 'linear' — no activation

    def parameters(self):
        return self.w + [self.b]

    def __repr__(self):
        return f'Neuron(n_in={len(self.w)}, act={self.activation!r})'


class Layer(Module):
    def __init__(self, n_in, n_out, **kwargs):
        self.neurons = [Neuron(n_in, **kwargs) for _ in range(n_out)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs  # unwrap single-output layers

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP(Module):
    def __init__(self, n_in, layer_sizes):
        dims = [n_in] + layer_sizes
        # hidden layers: tanh; output layer: linear
        self.layers = [
            Layer(dims[i], dims[i+1],
                  activation='tanh' if i < len(layer_sizes) - 1 else 'linear')
            for i in range(len(layer_sizes))
        ]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [ ]:
random.seed(0)
net = MLP(3, [4, 4, 1])
print(f'Parameters: {len(net.parameters())}')

out = net([1.0, -2.0, 0.5])
print(f'Forward pass: {out}')

# visualise a tiny MLP (2-in, 2 hidden, 1 out) — keeps the graph readable
random.seed(0)
tiny = MLP(2, [2, 1])
y_hat = tiny([Value(1.0, label='x1'), Value(-1.0, label='x2')])
y_hat.label = 'out'
draw_dot(y_hat)

## Part 5 — Training loop

**Every epoch:**
```
forward   → compute predictions and loss
zero_grad → clear .grad on every parameter   ← easy to forget
backward  → fill in all .grad values
SGD step  → p.data -= lr * p.grad
```

We train on **XOR** with targets ±1 and MSE loss.
A linear output neuron with MSE will push predictions toward ±1.

In [ ]:
random.seed(42)
model = MLP(2, [8, 8, 1])
print(f'Parameters: {len(model.parameters())}')

# XOR: output is 1 when inputs differ, -1 when they match
X = [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]
y = [-1.0,         1.0,        1.0,        -1.0]

losses = []
for epoch in range(400):
    preds = [model(xi) for xi in X]
    loss  = sum((p - yi)**2 for p, yi in zip(preds, y)) * (1.0 / len(y))

    model.zero_grad()   # reset grads before backward
    loss.backward()

    lr = 0.05
    for p in model.parameters():
        p.data -= lr * p.grad

    losses.append(loss.data)
    if epoch % 50 == 0:
        print(f'epoch {epoch:3d}  loss = {loss.data:.6f}')

print()
print('Final predictions:')
for xi, yi in zip(X, y):
    pred = model(xi)
    print(f'  {xi}  target={yi:+.0f}  pred={pred.data:+.4f}')

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(losses, linewidth=1.5, color='steelblue')
plt.xlabel('epoch')
plt.ylabel('MSE loss')
plt.title('XOR Training Loss')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# forward pass through the trained model on one input — shows the full graph
sample   = [Value(0.0, label='x1'), Value(1.0, label='x2')]
pred     = model(sample)
pred.label = 'pred'
draw_dot(pred)   # large graph — zoom in to see individual weight nodes